# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font> </center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 5 — Evaluación de Image Enhancement sobre Modelos de Profundidad Endoscópica Entrenados en SCARED</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### Diferencia respecto a Avance 4

En el **Avance 4** se evaluaron modelos con sus pesos **originales del paper** (entrenados en Hamlyn, KITTI o Endovis).

En este **Avance 5** se usan pesos **re-entrenados en SCARED** (proporcionados por el Dr. Ricardo Espinosa Loera), evaluados en los mismos datasets 8–9. Esto responde la pregunta: **¿cambia el impacto del enhancement cuando el modelo conoce el dominio de evaluación?**

### Hipótesis
El fine-tuning en SCARED mejora el baseline (none) de todos los modelos. El efecto del enhancement se reduce para los modelos que ya manejan bien las especularidades del dominio.


---
## 0. Pipeline general

```mermaid
flowchart TD
    A["SCARED - datasets 8-9, keyframes 0-4"] --> B
    B["Imagen RGB 1280x1024 px"] --> C1 & C2 & C3 & C4

    subgraph ENHANCEMENT ["Image Enhancement"]
        C1["None - baseline"]
        C2["Retinex SSR - Rahman 2004"]
        C3["EndoLMSPEC - Endo4IE - Garcia-Vega 2022"]
        C4["IAT EndoViT - Endo4IE - Wang 2022"]
    end

    C1 & C2 & C3 & C4 --> D1 & D2 & D3 & D4 & D5 & D6

    subgraph DEPTH ["Modelos — pesos SCARED (Dr. Espinosa Loera)"]
        D1["Monodepth2 - ResNet18 - epoch 19"]
        D2["MonoViT - MPViT-Small - epoch 19"]
        D3["EndoSfMLearner - DispResNet18"]
        D4["AF-SfMLearner - ResNet18 + AppFlow"]
        D5["MonoIIT - MPViT + Lighting - winner"]
        D6["weights19-MonoViT - MPViT + Lighting"]
    end

    D1 & D2 & D3 & D4 & D5 & D6 --> E["Median Scaling a mm"]
    E --> F["AbsRel - SqRel - RMSE - RMSELog - delta - FPS"]

    style ENHANCEMENT fill:#fff8e1,stroke:#E0A800,stroke-width:2px
    style DEPTH fill:#e8f4fd,stroke:#2766CB,stroke-width:2px
```

**Diseño factorial**: 4 enhancements × 6 modelos × 10 keyframes = **240 evaluaciones**


In [4]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "tifffile", "scikit-image", "fvcore"])

import torch, tifffile, cv2
import numpy as np
print(f"torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")


torch  : 2.11.0+cu128
CUDA   : True


---
## 1. Configuración de rutas

### Estructura esperada en Google Drive

```
MyDrive/proyecto_integrador/
├── scared_raw/
├── Endo-Depth-and-Motion/        ← código Monodepth2 / Endo-Depth
├── EndoLMSPEC/
├── EndoViT/
├── EndoSLAM/
├── MonoViT/                      ← código MonoViT (también para MonoIIT)
├── AF-SfMLearner/                ← código AF-SfMLearner
└── scared weights/               ← pesos SCARED (Dr. Espinosa Loera)
    ├── monodepth2_weights/weights_19/                ← encoder.pth + depth.pth
    ├── monovit_weights/weights_19/                   ← encoder.pth + depth.pth
    ├── endosfmlearner_weights/11-09-03_58/           ← dispnet_model_best.pth.tar
    ├── afmlearner_weights/Model_trained_end_to_end/  ← encoder.pth + depth.pth
    ├── monoIIT_weights/trained-winner-weights/       ← encoder.pth + depth.pth (+ lighting.pth)
    ├── weights_19_MonoViT/weights_19/                ← encoder.pth + depth.pth (+ lighting.pth)
    └── endodac_weights/  (pendiente — código propio)
```

> **Nota sobre MonoIIT**: el Dr. Espinosa indicó que los pesos de MonoIIT
> se ejecutan con el código de testing de MonoViT (misma arquitectura MPViT,
> con un módulo de iluminación adicional que no se usa en inferencia de profundidad).

### Tabla de pesos — Avance 5 vs Avance 4

| Modelo | Avance 4 (pesos originales) | Avance 5 (pesos SCARED) |
|---|---|---|
| Monodepth2 | Hamlyn | SCARED |
| MonoViT | KITTI | SCARED |
| EndoSfMLearner | EndoSLAM | SCARED |
| AF-SfMLearner | Endovis (paper) | SCARED |
| MonoIIT | — (no en A4) | SCARED |
| weights19-MonoViT | — (no en A4) | SCARED |

In [5]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE          = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT   = BASE / "scared_raw"
    EDAM_PATH     = BASE / "Endo-Depth-and-Motion"    # codigo Monodepth2
    LMSPEC_PATH   = BASE / "EndoLMSPEC"
    IAT_PATH      = BASE / "EndoViT"
    ENDOSLAM_PATH = BASE / "EndoSLAM"
    MONOVIT_PATH  = BASE / "MonoViT"
    AFSFM_PATH    = BASE / "AF-SfMLearner"

    # Pesos SCARED (Dr. Espinosa) — carpeta "scared weights" con nombres/subcarpetas originales
    W = BASE / "scared weights"
    W_MONO2   = W / "monodepth2_weights" / "weights_19"
    W_MONOVIT = W / "monovit_weights" / "weights_19"
    W_ENDOSFM = W / "endosfmlearner_weights" / "11-09-03_58"
    W_AFSFM   = W / "afmlearner_weights" / "Model_trained_end_to_end"
    W_MONOIIT = W / "monoIIT_weights" / "trained-winner-weights"
    W_W19MONO = W / "weights_19_MonoViT" / "weights_19"

    REPO_ROOT = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git",
            str(REPO_ROOT)
        ])
    subprocess.check_call(["git", "-C", str(REPO_ROOT), "config",
                           "user.email", "jmtoralcruz@gmail.com"])
    subprocess.check_call(["git", "-C", str(REPO_ROOT), "config",
                           "user.name", "jmtoral"])
    OUT_DIR = REPO_ROOT / "outcomes" / "avance5"

else:
    BASE          = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT   = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH     = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH   = Path("E:/EndoLMSPEC")
    IAT_PATH      = Path("E:/EndoVit")
    ENDOSLAM_PATH = Path("E:/EndoSLAM")
    MONOVIT_PATH  = Path("E:/MonoViT")
    AFSFM_PATH    = Path("E:/AF-SfMLearner")

    W = BASE
    W_MONO2   = W / "monodepth2_weights" / "weights_19"
    W_MONOVIT = W / "monovit_weights" / "weights_19"
    W_ENDOSFM = W / "endosfmlearner_weights" / "11-09-03_58"
    W_AFSFM   = W / "afmlearner_weights" / "Model_trained_end_to_end"
    W_MONOIIT = W / "monoIIT_weights" / "trained-winner-weights"
    W_W19MONO = W / "weights_19_MonoViT" / "weights_19"

    REPO_ROOT = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    OUT_DIR   = REPO_ROOT / "outcomes" / "avance5"

LMSPEC_WEIGHTS  = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS     = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
ENDOSFM_WEIGHTS = W_ENDOSFM / "dispnet_model_best.pth.tar"

OUT_DIR.mkdir(parents=True, exist_ok=True)
CAP_MM = 150.0
EVAL_KEYFRAMES = [
    ("dataset_8","keyframe_0"),("dataset_8","keyframe_1"),
    ("dataset_8","keyframe_2"),("dataset_8","keyframe_3"),
    ("dataset_8","keyframe_4"),("dataset_9","keyframe_0"),
    ("dataset_9","keyframe_1"),("dataset_9","keyframe_2"),
    ("dataset_9","keyframe_3"),("dataset_9","keyframe_4"),
]

print(f"Entorno : {'Colab' if IN_COLAB else 'Local'}")
print(f"OUT_DIR : {OUT_DIR}")
for n, p in [("SCARED_ROOT",SCARED_ROOT),("LMSPEC_WEIGHTS",LMSPEC_WEIGHTS),
             ("IAT_WEIGHTS",IAT_WEIGHTS),("ENDOSFM_WEIGHTS",ENDOSFM_WEIGHTS),
             ("W_MONO2",W_MONO2),("W_MONOVIT",W_MONOVIT),
             ("W_AFSFM",W_AFSFM),("W_MONOIIT",W_MONOIIT),
             ("W_W19MONO",W_W19MONO)]:
    print(f"  {n:20s}: {'✓' if Path(p).exists() else '✗ NO ENCONTRADO'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno : Colab
OUT_DIR : /content/repo_52/outcomes/avance5
  SCARED_ROOT         : ✓
  LMSPEC_WEIGHTS      : ✓
  IAT_WEIGHTS         : ✓
  ENDOSFM_WEIGHTS     : ✓
  W_MONO2             : ✓
  W_MONOVIT           : ✓
  W_AFSFM             : ✓
  W_MONOIIT           : ✓
  W_W19MONO           : ✓


---
## 2. Cargar Modelos — Pesos SCARED

### 2a. Monodepth2 (ResNet-18, entrenado en SCARED)

In [6]:
import torch
import numpy as np

sys.path.insert(0, str(EDAM_PATH / "apps" / "depth_estimate"))
from resnet_encoder import ResnetEncoder
from depth_decoder import DepthDecoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

mono2_encoder = ResnetEncoder(18, False)
enc_dict = torch.load(W_MONO2 / "encoder.pth", map_location=DEVICE)
MONO2_H = enc_dict.get("height", 192)
MONO2_W = enc_dict.get("width",  640)
filtered = {k: v for k, v in enc_dict.items() if k in mono2_encoder.state_dict()}
mono2_encoder.load_state_dict(filtered)
mono2_encoder.to(DEVICE).eval()

mono2_decoder = DepthDecoder(num_ch_enc=mono2_encoder.num_ch_enc, scales=range(4))
mono2_decoder.load_state_dict(torch.load(W_MONO2 / "depth.pth", map_location=DEVICE))
mono2_decoder.to(DEVICE).eval()

print(f"Monodepth2 (SCARED): {MONO2_H}x{MONO2_W} — cargado OK")


Dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Monodepth2 (SCARED): 256x320 — cargado OK


### 2b. MonoViT (MPViT-Small, entrenado en SCARED)

In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm", "einops"])

sys.path.insert(0, str(MONOVIT_PATH))
import networks as mvnet

# Patrón oficial de MonoViT/evaluate_depth.py: encoder mpvit_small + DepthDecoder
# separados (NO DeepNet). Los pesos SCARED del Dr. Espinosa usan esta estructura.
def load_monovit_pair(weights_dir, device):
    encoder = mvnet.mpvit_small()
    encoder.num_ch_enc = [64, 128, 216, 288, 288]
    enc_dict = torch.load(weights_dir / "encoder.pth", map_location=device)
    h = enc_dict.get("height", 192); w = enc_dict.get("width", 640)
    model_dict = encoder.state_dict()
    encoder.load_state_dict({k: v for k, v in enc_dict.items() if k in model_dict})
    encoder.to(device).eval()

    decoder = mvnet.DepthDecoder()
    decoder.load_state_dict(torch.load(weights_dir / "depth.pth", map_location=device))
    decoder.to(device).eval()
    return encoder, decoder, h, w

monovit_enc, monovit_dec, MONOVIT_H, MONOVIT_W = load_monovit_pair(W_MONOVIT, DEVICE)
print(f"MonoViT (SCARED): {MONOVIT_H}x{MONOVIT_W} — cargado OK")

### 2c. EndoSfMLearner (DispResNet-18, entrenado en SCARED)

In [ ]:
import importlib.util

_endosfm_dir = ENDOSLAM_PATH / "EndoSfMLearner"
if str(_endosfm_dir) not in sys.path:
    sys.path.insert(0, str(_endosfm_dir))

_spec = importlib.util.spec_from_file_location(
    "_endosfm_models_a5",
    str(_endosfm_dir / "models" / "__init__.py"),
    submodule_search_locations=[str(_endosfm_dir / "models")]
)
endosfm_models_a5 = importlib.util.module_from_spec(_spec)
sys.modules["_endosfm_models_a5"] = endosfm_models_a5
_spec.loader.exec_module(endosfm_models_a5)

endosfm_scared = endosfm_models_a5.DispResNet(18, False).to(DEVICE)
w = torch.load(ENDOSFM_WEIGHTS, map_location=DEVICE)
endosfm_scared.load_state_dict(w["state_dict"])
endosfm_scared.eval()

n = sum(p.numel() for p in endosfm_scared.parameters())
print(f"EndoSfMLearner (SCARED): {n/1e6:.2f} M — cargado OK")


### 2d. AF-SfMLearner (ResNet-18 + Appearance Flow, entrenado en SCARED)

In [ ]:
# AF-SfMLearner usa la misma arquitectura que Endo-Depth (ResnetEncoder + DepthDecoder)
afsfm_scared_enc = ResnetEncoder(18, False)
enc_w = torch.load(W_AFSFM / "encoder.pth", map_location=DEVICE)
AFSFM_H = enc_w.get("height", 256)
AFSFM_W = enc_w.get("width",  320)
filtered = {k: v for k, v in enc_w.items() if k in afsfm_scared_enc.state_dict()}
afsfm_scared_enc.load_state_dict(filtered)
afsfm_scared_enc.to(DEVICE).eval()

afsfm_scared_dec = DepthDecoder(num_ch_enc=afsfm_scared_enc.num_ch_enc, scales=range(4))
afsfm_scared_dec.load_state_dict(torch.load(W_AFSFM / "depth.pth", map_location=DEVICE))
afsfm_scared_dec.to(DEVICE).eval()

print(f"AF-SfMLearner (SCARED): {AFSFM_H}x{AFSFM_W} — cargado OK")


### 2e. MonoIIT (MPViT + Lighting, entrenado en SCARED)

**Nota del Dr. Espinosa Loera**: los pesos de MonoIIT se ejecutan con el código de MonoViT.
La arquitectura base es MPViT-Small (igual que MonoViT). El módulo de iluminación (`lighting.pth`)
se usa durante el entrenamiento; para inferencia de profundidad se usan solo `encoder.pth` + `depth.pth`.


In [ ]:
# MonoIIT: misma arquitectura/código que MonoViT, pesos diferentes
monoIIT_enc, monoIIT_dec, MONOIIT_H, MONOIIT_W = load_monovit_pair(W_MONOIIT, DEVICE)
print(f"MonoIIT (SCARED): {MONOIIT_H}x{MONOIIT_W} — cargado OK")
if (W_MONOIIT / "lighting.pth").exists():
    print("  lighting.pth presente (no usado en inferencia de profundidad)")

### 2f. weights19-MonoViT 

Esta carpeta tiene la misma estructura que MonoIIT (con `lighting.pth`).
Podría ser MonoIIF, el tercer modelo propuesto. **Pendiente de confirmación.**


In [ ]:
# weights_19_MonoViT: misma estructura, posible MonoIIF
w19_enc, w19_dec, W19_H, W19_W = load_monovit_pair(W_W19MONO, DEVICE)
print(f"weights19-MonoViT (SCARED): {W19_H}x{W19_W} — cargado OK")
if (W_W19MONO / "lighting.pth").exists():
    print("  lighting.pth presente — probable MonoIIF (confirmar con Dr. Espinosa)")

---
## 3. Métodos de Image Enhancement

Idénticos al Avance 4 — mismas funciones de corrección.

In [ ]:
import cv2, numpy as np, torch, torchvision.transforms as T
import importlib.util, types

# EndoLMSPEC
def _load_endolmspec(lmspec_path, device):
    _orig = sys.path.copy()
    clean = [str(lmspec_path)] + [
        p for p in sys.path
        if "EndoSLAM" not in p and "endosfm" not in p.lower()
        and "HADepth" not in p]
    for k in list(sys.modules):
        if k in ("utils","generator","unet") or k.startswith("utils."): del sys.modules[k]
    try:
        sys.path = clean
        spec = importlib.util.spec_from_file_location("generator", lmspec_path/"generator.py")
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        Generator = mod.Generator
    finally:
        sys.path = _orig
    return Generator(n_channels=3, device=device, bilinear=False), Generator

lmspec_net, _ = _load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE))
lmspec_net.to(DEVICE).eval()
print("EndoLMSPEC OK")

# IAT
sys.modules["imp"] = types.ModuleType("imp")
_iat_model_path = IAT_PATH / "experiments" / "model" / "IAT_main.py"
_spec = importlib.util.spec_from_file_location("IAT_main_a5", _iat_model_path)
_iat_mod = importlib.util.module_from_spec(_spec)
_iat_path = str(IAT_PATH / "experiments")
if _iat_path not in sys.path: sys.path.insert(0, _iat_path)
_spec.loader.exec_module(_iat_mod)
iat_net = _iat_mod.IAT(in_dim=3, with_global=True, type="exp")
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE))
iat_net.to(DEVICE).eval()
print("IAT OK")

def correct_none(img): return img

def correct_retinex(img, sigma=30):
    img_f = img.astype(np.float32) + 1.0
    result = np.zeros_like(img_f)
    for c in range(3):
        blur = cv2.GaussianBlur(img_f[:,:,c],(0,0),sigma)
        result[:,:,c] = np.log(img_f[:,:,c]) - np.log(blur+1.0)
    result -= result.min()
    return (result/(result.max()+1e-8)*255).astype(np.uint8)

def correct_endolmspec(img):
    t = T.ToTensor()(img).to(DEVICE)
    with torch.no_grad(): _, out = lmspec_net(t)
    return (out["subnet_16"][0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

def correct_iat(img):
    t = torch.from_numpy(img.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): _, _, enh = iat_net(t)
    return (enh[0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

CORRECTIONS = {"none":correct_none,"retinex":correct_retinex,
               "endolmspec":correct_endolmspec,"iat":correct_iat}
print(f"Enhancements: {list(CORRECTIONS.keys())}")


---
## 4. Funciones de inferencia y métricas

In [ ]:
import time, torch.nn.functional as F, PIL.Image as pil
from torchvision import transforms
from scipy.spatial import cKDTree
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.transform import resize as imresize

FX, FY, CX, CY = 1078.0, 1078.0, 640.0, 512.0

def _predict_monodepth2_style(img_rgb, enc, dec, h, w, device):
    """Monodepth2/AF-SfMLearner: ToTensor sin normalizacion ImageNet."""
    H, W = img_rgb.shape[:2]
    t = transforms.ToTensor()(pil.fromarray(img_rgb).resize((w,h),pil.LANCZOS)).unsqueeze(0).to(device)
    if device.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = dec(enc(t))
    if device.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    disp = F.interpolate(out[("disp",0)],(H,W),mode="bilinear",align_corners=False).squeeze().cpu().numpy()
    return 1.0/(1/100+(1/0.1-1/100)*disp), ms

def _predict_monovit_style(img_rgb, enc, dec, h, w, device):
    """MonoViT/MonoIIT: encoder mpvit_small + DepthDecoder separados.
    ToTensor sin normalizacion ImageNet (igual que evaluate_depth.py oficial)."""
    H, W = img_rgb.shape[:2]
    t = transforms.ToTensor()(pil.fromarray(img_rgb).resize((w,h),pil.LANCZOS)).unsqueeze(0).to(device)
    if device.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = dec(enc(t))
    if device.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    disp = F.interpolate(out[("disp",0)],(H,W),mode="bilinear",align_corners=False).squeeze().cpu().numpy()
    return 1.0/(1/100+(1/0.1-1/100)*disp), ms

def predict_mono2(img):     return _predict_monodepth2_style(img, mono2_encoder, mono2_decoder, MONO2_H, MONO2_W, DEVICE)
def predict_monovit(img):   return _predict_monovit_style(img, monovit_enc, monovit_dec, MONOVIT_H, MONOVIT_W, DEVICE)
def predict_afsfm(img):     return _predict_monodepth2_style(img, afsfm_scared_enc, afsfm_scared_dec, AFSFM_H, AFSFM_W, DEVICE)
def predict_monoIIT(img):   return _predict_monovit_style(img, monoIIT_enc, monoIIT_dec, MONOIIT_H, MONOIIT_W, DEVICE)
def predict_w19mono(img):   return _predict_monovit_style(img, w19_enc, w19_dec, W19_H, W19_W, DEVICE)

def predict_endosfm(img):
    H, W = img.shape[:2]
    r = imresize(img,(256,832)).astype(np.float32)
    t = torch.from_numpy(((r/255-0.45)/0.225).transpose(2,0,1)).unsqueeze(0).to(DEVICE)
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad(): d = endosfm_scared(t)
    if DEVICE.type=="cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-t0)*1000
    return 1/(imresize(d.squeeze().cpu().numpy(),(H,W))+1e-6), ms

DEPTH_MODELS = {
    "Monodepth2":      predict_mono2,
    "MonoViT":         predict_monovit,
    "EndoSfMLearner":  predict_endosfm,
    "AF-SfMLearner":   predict_afsfm,
    "MonoIIT":         predict_monoIIT,
    "w19-MonoViT":     predict_w19mono,
}

def specular_mask(img, pct=97, dil=15):
    L = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:,:,0].astype(np.float32)
    m = (L>=np.percentile(L,pct)).astype(np.uint8)
    return cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(dil,dil))).astype(bool)

def depth_to_pc(d, mask, fx=FX, fy=FY, cx=CX, cy=CY):
    H,W = d.shape; uu,vv = np.meshgrid(np.arange(W),np.arange(H))
    Z = d[mask]
    return np.stack([(uu[mask]-cx)*Z/fx,(vv[mask]-cy)*Z/fy,Z],axis=1)

def chamfer(p1,p2,n=50_000):
    if not len(p1) or not len(p2): return np.nan
    rng = np.random.default_rng(42)
    if len(p1)>n: p1=p1[rng.choice(len(p1),n,replace=False)]
    if len(p2)>n: p2=p2[rng.choice(len(p2),n,replace=False)]
    d1,_ = cKDTree(p2).query(p1); d2,_ = cKDTree(p1).query(p2)
    return float((d1.mean()+d2.mean())/2)

def compute_metrics(img_orig, img_corr, depth_rel, gt_mm, cap=150.0):
    valid = (~np.isnan(gt_mm))&(gt_mm>0)&(gt_mm<cap)
    if valid.sum()==0:
        return {k:np.nan for k in ["AbsRel","SqRel","RMSE","RMSELog",
                                   "delta_1","delta_2","delta_3",
                                   "Chamfer","AbsRel_spec","AbsRel_nospec","scale"]}
    scale = np.median(gt_mm[valid])/(np.median(depth_rel[valid])+1e-8)
    pred  = depth_rel*scale; d,gt = pred[valid],gt_mm[valid]
    spec  = specular_mask(img_orig); vs,vn = valid&spec, valid&~spec
    ratio = np.maximum(d/(gt+1e-8), gt/(d+1e-8))
    return {
        "scale":    round(float(scale),4),
        "AbsRel":   round(float(np.mean(np.abs(d-gt)/(gt+1e-8))),4),
        "SqRel":    round(float(np.mean((d-gt)**2/(gt+1e-8))),4),
        "RMSE":     round(float(np.sqrt(np.mean((d-gt)**2))),3),
        "RMSELog":  round(float(np.sqrt(np.mean((np.log(np.clip(d,1e-3,None))-np.log(np.clip(gt,1e-3,None)))**2))),4),
        "delta_1":  round(float(np.mean(ratio<1.25)),4),
        "delta_2":  round(float(np.mean(ratio<1.25**2)),4),
        "delta_3":  round(float(np.mean(ratio<1.25**3)),4),
        "Chamfer":  round(chamfer(depth_to_pc(np.where(valid,pred,np.nan),valid),
                                  depth_to_pc(np.where(valid,gt_mm,np.nan),valid)),3),
        "AbsRel_spec":  round(float(np.mean(np.abs(pred[vs]-gt_mm[vs])/(gt_mm[vs]+1e-8))),4) if vs.sum()>0 else np.nan,
        "AbsRel_nospec":round(float(np.mean(np.abs(pred[vn]-gt_mm[vn])/(gt_mm[vn]+1e-8))),4) if vn.sum()>0 else np.nan,
    }

print(f"Modelos  : {list(DEPTH_MODELS.keys())}")
print(f"Enhanc.  : {list(CORRECTIONS.keys())}")
print(f"Total    : {len(DEPTH_MODELS)*len(CORRECTIONS)*len(EVAL_KEYFRAMES)} evaluaciones")

---
## 5. Carga de datos SCARED

Mismo protocolo que Avances 3–4: datasets 8–9, keyframes 0–4.

In [ ]:
import io, zipfile, tifffile

def load_keyframe(scared_root, dataset_id, keyframe_id):
    with zipfile.ZipFile(scared_root/f"{dataset_id}.zip") as z:
        buf = np.frombuffer(z.read(f"{dataset_id}/{keyframe_id}/Left_Image.png"), np.uint8)
        img = cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        with z.open(f"{dataset_id}/{keyframe_id}/left_depth_map.tiff") as f:
            tiff = tifffile.imread(io.BytesIO(f.read()))
        depth_z = tiff[...,2].astype(np.float32)
        depth_z[depth_z<=0] = np.nan
    return img, depth_z

img_test, depth_test = load_keyframe(SCARED_ROOT, *EVAL_KEYFRAMES[0])
print(f"Imagen: {img_test.shape}  GT válido: {(~np.isnan(depth_test)).mean()*100:.1f}%")


---
## 6. Experimento factorial: 4 enhancements × 6 modelos × 10 keyframes

Total: **240 evaluaciones**

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

assert len(DEPTH_MODELS)==6, f"Se esperan 6 modelos, hay {len(DEPTH_MODELS)}"
results = []

for dataset_id, keyframe_id in tqdm(EVAL_KEYFRAMES, desc="Keyframes"):
    img_rgb, gt_mm = load_keyframe(SCARED_ROOT, dataset_id, keyframe_id)
    for corr_name, corr_fn in CORRECTIONS.items():
        t0 = time.perf_counter()
        img_corr = corr_fn(img_rgb)
        t_enh = (time.perf_counter()-t0)*1000
        for model_name, depth_fn in DEPTH_MODELS.items():
            times = [depth_fn(img_corr)[1] for _ in range(3)]
            depth_rel, _ = depth_fn(img_corr)
            t_inf = float(np.median(times))
            m = compute_metrics(img_rgb, img_corr, depth_rel, gt_mm, CAP_MM)
            results.append({
                "Dataset":dataset_id,"Keyframe":keyframe_id,
                "Modelo":model_name,"Método":corr_name,
                "T_enhance_ms":round(t_enh,1),"T_depth_ms":round(t_inf,1),
                "T_total_ms":round(t_enh+t_inf,1),
                "FPS":round(1000/(t_enh+t_inf),1),
                **m,
            })
            print(f"  {dataset_id}/{keyframe_id} | {model_name:16s} | {corr_name:12s} | "
                  f"AbsRel={m['AbsRel']:.4f}")

df = pd.DataFrame(results)
df.to_csv(OUT_DIR/"avance5_results.csv", index=False)
print(f"\nExperimento completo — {len(df)} evaluaciones")


---
## 7. Resumen de resultados

In [ ]:
numeric_cols = ["AbsRel","SqRel","RMSE","RMSELog",
                "delta_1","delta_2","delta_3",
                "AbsRel_spec","AbsRel_nospec","T_total_ms","FPS"]

summary = (df.groupby(["Modelo","Método"])[numeric_cols]
             .mean().round(4)
             .sort_values(["Modelo","AbsRel"]))

for model in df["Modelo"].unique():
    sub = summary.loc[model]
    if "none" in sub.index:
        b = sub.loc["none","AbsRel"]
        summary.loc[(model,slice(None)),"ΔAbsRel_%"] = (
            (summary.loc[(model,slice(None)),"AbsRel"]-b)/b*100).round(1)

# Compatibilidad
if "FPS" not in summary.columns:
    summary["FPS"] = df.groupby(["Modelo","Método"])["FPS"].mean().round(1)

print("="*115)
print(f"{'Modelo':<18} {'Enhancement':<12} {'AbsRel':>7} {'SqRel':>7} {'RMSE':>7} "
      f"{'RMSELog':>8} {'d1.25':>6} {'d1.25^2':>8} {'d1.25^3':>8} "
      f"{'IT(ms)':>7} {'ΔAR%':>6}")
print("-"*115)
for model in df["Modelo"].unique():
    for method in ["none","retinex","endolmspec","iat"]:
        if (model,method) not in summary.index: continue
        row = summary.loc[(model,method)]
        delta = f"{row['ΔAbsRel_%']:+.1f}%" if method!="none" else "base"
        mark  = " ←" if method!="none" and row["AbsRel"]<summary.loc[(model,"none"),"AbsRel"] else ""
        print(f"  {model:<16} {method:<12} "
              f"{row['AbsRel']:>7.4f} {row['SqRel']:>7.4f} {row['RMSE']:>7.3f} "
              f"{row['RMSELog']:>8.4f} {row['delta_1']:>6.4f} {row['delta_2']:>8.4f} "
              f"{row['delta_3']:>8.4f} {row['T_total_ms']:>7.1f} {delta:>6}{mark}")
    print()
print("ΔAR% negativo = mejora vs. baseline (none)")


---
## 8. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt

models_list  = list(df["Modelo"].unique())
methods_list = list(df["Método"].unique())
colors = {"none":"#8aa0c0","retinex":"#4599EC","endolmspec":"#E0A800","iat":"#2ecc71"}
x = np.arange(len(methods_list))

metrics_plot = [
    ("AbsRel",      "AbsRel (↓ mejor)",           "Primaria"),
    ("RMSE",        "RMSE mm (↓ mejor)",            "Geométrica"),
    ("AbsRel_spec", "AbsRel especular (↓ mejor)",   "Hipótesis"),
    ("FPS",         "FPS total (↑ mejor)",           "Clínica"),
]
n_m, n_met = len(models_list), len(metrics_plot)

fig, axes = plt.subplots(n_m, n_met, figsize=(4*n_met, 4*n_m))
fig.suptitle("Avance 5 — Enhancement × Modelos SCARED\n(promedio 10 keyframes)",
             fontsize=13, fontweight="bold")

for r, model in enumerate(models_list):
    sub = summary.loc[model] if model in summary.index.get_level_values(0) else None
    for c, (metric, label, cat) in enumerate(metrics_plot):
        ax = axes[r,c] if n_m>1 else axes[c]
        vals = [sub.loc[m,metric] if sub is not None and m in sub.index else np.nan
                for m in methods_list]
        bars = ax.bar(x, vals, 0.62, color=[colors[m] for m in methods_list])
        ax.set_xticks(x); ax.set_xticklabels(methods_list, fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        if r==0: ax.set_title(f"{label}\n[{cat}]", fontsize=9)
        if c==0: ax.set_ylabel(model, fontsize=10, fontweight="bold")
        top = max(v for v in vals if not np.isnan(v)) if any(not np.isnan(v) for v in vals) else 1
        for b,v in zip(bars,vals):
            if not np.isnan(v):
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+top*0.015,
                        f"{v:.3f}" if metric!="FPS" else f"{v:.0f}",
                        ha="center", va="bottom", fontsize=7)
        ax.set_ylim(0, top*1.22)

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig(OUT_DIR/"avance5_metricas.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
VIZ_KEYFRAMES = [EVAL_KEYFRAMES[0], EVAL_KEYFRAMES[5]]
CMAP = "plasma_r"

for ds_id, kf_id in VIZ_KEYFRAMES:
    img_rgb, gt_mm = load_keyframe(SCARED_ROOT, ds_id, kf_id)
    valid_gt = (~np.isnan(gt_mm))&(gt_mm>0)&(gt_mm<CAP_MM)
    corr_names = list(CORRECTIONS.keys())
    n_cols = 1+len(corr_names)+1
    fig, axes = plt.subplots(len(DEPTH_MODELS), n_cols,
                             figsize=(3.5*n_cols, 3.5*len(DEPTH_MODELS)))

    cache = {}
    for model_name, depth_fn in DEPTH_MODELS.items():
        for cn, cf in CORRECTIONS.items():
            ic = cf(img_rgb); dr, _ = depth_fn(ic)
            sc = np.median(gt_mm[valid_gt])/(np.median(dr[valid_gt])+1e-8)
            cache[(model_name,cn)] = (ic, dr, dr*sc)

    for row,(model_name,_) in enumerate(DEPTH_MODELS.items()):
        ax_row = axes[row]
        ax_row[0].imshow(img_rgb); ax_row[0].axis("off")
        ax_row[0].set_ylabel(model_name, fontsize=8, fontweight="bold")
        if row==0: ax_row[0].set_title("RGB", fontsize=8)

        for col, cn in enumerate(corr_names, 1):
            ic, dr, pm = cache[(model_name,cn)]
            m = compute_metrics(img_rgb, ic, dr, gt_mm, CAP_MM)
            v1,v2 = np.nanpercentile(pm[valid_gt],2), np.nanpercentile(pm[valid_gt],98)
            ax_row[col].imshow(pm, cmap=CMAP, vmin=v1, vmax=v2); ax_row[col].axis("off")
            if row==0: ax_row[col].set_title(f"{cn}\nAbsRel={m['AbsRel']:.4f}", fontsize=7)
            else:      ax_row[col].set_title(f"AbsRel={m['AbsRel']:.4f}", fontsize=7)

        v1,v2 = np.nanpercentile(gt_mm[valid_gt],2), np.nanpercentile(gt_mm[valid_gt],98)
        ax_row[-1].imshow(gt_mm, cmap=CMAP, vmin=v1, vmax=v2); ax_row[-1].axis("off")
        if row==0: ax_row[-1].set_title("GT (mm)", fontsize=8)

    plt.suptitle(f"Avance 5 — {ds_id}/{kf_id}  (plasma_r: amarillo=cerca, azul=lejos)",
                 fontsize=9, y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR/f"avance5_viz_{ds_id}_{kf_id}.png", dpi=110, bbox_inches="tight")
    plt.show()


---
## 9. Comparativa Avance 4 vs. Avance 5 (baseline none)

La siguiente celda carga los resultados del Avance 4 (si están en Drive/repo)
y compara el baseline `none` de cada modelo entre pesos originales y pesos SCARED.


In [ ]:
# Cargar resultados de Avance 4 si existen
a4_csv = REPO_ROOT / "outcomes" / "avance4" / "avance4_results.csv"
if a4_csv.exists():
    df4 = pd.read_csv(a4_csv)
    base4 = (df4[df4["Método"]=="none"]
               .groupby("Modelo")[["AbsRel","RMSE","delta_1","FPS"]]
               .mean().round(4))
    base4.columns = [f"{c}_A4" for c in base4.columns]

    base5 = (df[df["Método"]=="none"]
               .groupby("Modelo")[["AbsRel","RMSE","delta_1","FPS"]]
               .mean().round(4))
    base5.columns = [f"{c}_A5" for c in base5.columns]

    comp = base4.join(base5, how="outer")
    # Delta: A5 vs A4
    for col in ["AbsRel","RMSE"]:
        comp[f"Δ{col}"] = ((comp[f"{col}_A5"]-comp[f"{col}_A4"])/comp[f"{col}_A4"]*100).round(1)

    print("Comparativa baseline (none): A4 (pesos originales) vs A5 (pesos SCARED)")
    print("Δ negativo = A5 es mejor\n")
    print(comp.to_string())
else:
    print(f"No se encontró {a4_csv}")
    print("Ejecuta primero el Avance 4 para tener resultados comparativos.")


---
## 10. Exportar resultados al repositorio

Las imágenes se guardan en `outcomes/avance5/`.


In [ ]:
import subprocess, shutil

def git(args):
    subprocess.check_call(["git","-C",str(REPO_ROOT)]+args)

if IN_COLAB:
    from getpass import getpass
    token = getpass("GitHub token (repo scope): ")
    git(["remote","set-url","origin",
         f"https://{token}@github.com/jmtoral/proyecto_integrador_52.git"])
    git(["pull","--rebase"])

new_files = list(OUT_DIR.glob("avance5_*.png"))
csv_file  = OUT_DIR/"avance5_results.csv"
if csv_file.exists(): new_files.append(csv_file)

print(f"OUT_DIR  : {OUT_DIR}")
print(f"Archivos : {[f.name for f in new_files]}")

if new_files:
    git(["add"]+[str(f.relative_to(REPO_ROOT)) for f in new_files])
    ch = subprocess.run(["git","-C",str(REPO_ROOT),"diff","--cached","--name-only"],
                        capture_output=True,text=True).stdout.strip()
    if ch:
        git(["commit","-m",f"Add: resultados Avance5 ({len(new_files)} archivos)"])
        git(["push"])
        print(f"Push OK: {[f.name for f in new_files]}")
    else:
        print("Sin cambios nuevos")
